# Welcome to the RL Robotics Workshop!
* We will be using the `panda_gym` library, which provides a collection of environments for simulating the Franka Emika Panda robotic arm.
The library is designed to be compatible with the `gymnasium` library, allowing us to easily create and interact with the environments.
* We will start with a simple environment called `PandaReach-v3`, where the goal is to move the robot's gripper to a target position in 3D space. We will explore how to manually control the arm and observe its movement.
* After that, we will dive into reinforcement learning algorithms to train the robot to reach the target position autonomously. We will use popular algorithms like PPO and SAC from the `stable_baselines3` library to achieve this. The environment this time will be `PandaPickAndPlace-v3`, where the robot must pick up an object and place it at a target location.

### Let's get started by installing the necessary libraries and setting up our environment!

In [1]:
import os

os.environ['__NV_PRIME_RENDER_OFFLOAD'] = '1'
os.environ['__GLX_VENDOR_LIBRARY_NAME'] = 'nvidia'

import time
import gymnasium as gym
import panda_gym
import numpy as np

from stable_baselines3 import A2C, PPO, SAC

import urllib.request

print(f"Gymnasium version: {gym.__version__}")

Gymnasium version: 1.2.3


## Our first environment: PandaReach-v3
PandaReach-v3 is a simple environment where a robotic arm (the Panda) must move its gripper to a target position in 3D space.

The agent we're going to use is a robotic arm that needs to do controls (moving the arm and using the end-effector).

In robotics, the *end-effector* is the device at the end of a robotic arm designed to interact with the environment.

In `PandaReach`, the robot must place its end-effector at a target position (__green ball__).

We're going to use both dense and sparse version of this environment. It means we'll get a *dense reward function* that **will provide a reward at each timestep** (the closer the agent is to completing the task, the higher the reward). Contrary to a *sparse reward function* where the environment **return a reward if and only if the task is completed**.

Also, we're going to use the *End-effector displacement control*, it means the **action corresponds to the displacement of the end-effector**. We don't control the individual motion of each joint (joint control).


![Panda Reach V3](assets/panda_robot.png)


#### Let's create the environment and see how it works!
#### How to interact with the environment:
* `CRTL + LEFT MOUSE BUTTON` to rotate the camera
* `CRTL + MIDDLE MOUSE BUTTON` to pan the camera
* `CRTL + RIGHT MOUSE BUTTON` to zoom in and out or use the scroll wheel to zoom in and out
* `w` Switches view to wireframe mode
* `g` toggles the side panels and the on-screen GUI overlay off and on. Pressing `g`to hide the clutter gives students a clean view of just the arm.
* `s` and `v` toggle other render debug modes (shadows, rendering details).
* `Esc` to exit the environment

In [2]:
# Create the environment with a live 3D window
env = gym.make("PandaReach-v3", render_mode="human")

pybullet build time: Jun  5 2026 12:56:14


argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=5
argv[0] = --unused
argv[1] = --background_color_red=0.8745098114013672
argv[2] = --background_color_green=0.21176470816135406
argv[3] = --background_color_blue=0.1764705926179886
argv[4] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA GeForce RTX 2050/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 610.43.02
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
pthread_getconcurrency()=0
Version = 3.3.0 NVIDIA 610.43.02
Vendor = NVIDIA Corporation
Renderer = NVIDIA GeForce RTX 2050/PCIe/SSE2
b3Pr

Let's randomly sample some actions to see how the arm moves. This will give us a feel for the environment and how the robot responds to different inputs. We will run a loop where we take random actions and observe the resulting movement of the arm in the 3D window. This is a great way to get familiar with the action space and the dynamics of the environment before we start training any reinforcement learning algorithms.

__NOTE: Don't worry about what each of the random actions means right now, we will dive into that in more detail later. For now, just enjoy watching the arm move around!__

In [16]:
for _ in range(60):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        env.reset()
    time.sleep(0.1)

Close the environment to free up the rendering window. This is important to ensure that we don't have multiple windows open and to free up system resources. After closing the environment, we can create it again if we want to continue experimenting with it or move on to a different environment for training our reinforcement learning algorithms.

In [3]:
# Close the environment to free up the rendering window
env.close()

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed


## What does each of the variables returned mean?
* `obs`: The observation returned by the environment after taking an action. It typically includes information about the current state of the environment, such as the position of the robot's gripper and the target position.
* `info`: Additional information about the environment after taking the action, which can include details about the current episode, such as the distance to the target or other relevant metrics. This information can be useful for debugging and analyzing the agent's performance.
* `action`: The action taken by the agent, which in this case is a random sample from the action space. It represents the movement commands sent to the robot's arm.
* `reward`: The reward received after taking the action, which is a numerical value that indicates how well the action performed in terms of achieving the goal. In the PandaReach environment, the reward is typically based on the distance between the gripper and the target position, with higher rewards for being closer to the target.
* `terminated`: A boolean value that indicates whether the episode has ended. An episode can end either because the agent has successfully reached the target or because it has failed to do so within a certain number of steps.
* `truncated`: A boolean value that indicates whether the episode was truncated, which can happen if the agent takes too long to reach the target or if it violates certain constraints. Truncation is a way to prevent the agent from getting stuck in an infinite loop of taking actions without making progress towards the goal.


Let's inspect the environment more closely by printing out the initial observation, the random action taken, and the resulting reward and info after taking that action. This will help us understand what the observation contains, how the action affects the environment, and what kind of feedback we get from the environment in terms of rewards and additional information. By analyzing these outputs, we can gain insights into how the environment works and how we might want to structure our reinforcement learning algorithms to achieve better performance in reaching the target position.

In [44]:
env = gym.make("PandaReachDense-v3")
s_size = env.observation_space.shape

print("\n===========================================================")
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample())  # Get a random observation

# Get the state space and action space
a_size = env.action_space
action = env.action_space.sample()
print("===========================================================")
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Random Action Sampled:", action)

env.reset()
obs, reward, terminated, truncated, info = env.step(action)
print("===========================================================")
print("\n _____ENVIRONMENT STEP_____ \n")
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Info:", info)
print("=============================================================\n")
env.close()

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886

_____OBSERVATION SPACE_____ 

The State Space is:  None
Sample observation {'achieved_goal': array([-6.6233473,  2.911893 ,  2.2864232], dtype=float32), 'desired_goal': array([ 1.79002  , -8.369694 ,  7.2449694], dtype=float32), 'observation': array([-4.2484527,  6.9203353, -2.1297846,  6.273937 ,  1.1629164,
        7.8398304], dtype=float32)}

 _____ACTION SPACE_____ 

The Action Space is:  Box(-1.0, 1.0, (3,), float32)
Random Action Sampled: [-0.51516557  0.7274916   0.5567973 ]
Reward: -0.05323180928826332
Terminated: False
Truncated: False
Info: {'is_success': False}



## Manual Control of the Arm

Now that we have a better understanding of the environment and how to interact with it, let's try to manually control the arm to reach the target position. We will use a simple proportional controller that calculates the direction from the current gripper position to the target position and generates actions that move the arm in that direction. This will allow us to see how we can use the observations from the environment to guide our actions and achieve the goal of reaching the target position. By doing this, we can also get a sense of how the reward structure works and how we can design our reinforcement learning algorithms to learn from these interactions effectively.

In [20]:
env = gym.make("PandaReach-v3", render_mode="human")
obs, info = env.reset()

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=5
argv[0] = --unused
argv[1] = --background_color_red=0.8745098114013672
argv[2] = --background_color_green=0.21176470816135406
argv[3] = --background_color_blue=0.1764705926179886
argv[4] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA GeForce RTX 2050/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 610.43.02
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
pthread_getconcurrency()=0
Version = 3.3.0 NVIDIA 610.43.02
Vendor = NVIDIA Corporation
Renderer = NVIDIA GeForce RTX 2050/PCIe/SSE2
b3Pr

### 1. Baseline Movement - Doing nothing


In [7]:
for _ in range(100):
    # Action array: [X, Y, Z]
    # 0.0 means "don't move"
    action = np.array([0.0, 0.0, 0.0])

    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.05) # Slow down the loop so we can watch

### 2. X-Axis Movement - Forward and Backward

In [21]:
env.reset()
# Move FORWARD (X-axis)
for _ in range(10):
    # action[0] is X. A positive number moves it forward.
    action = np.array([1.0, 0.0, 0.0])
    env.step(action)
    time.sleep(0.1)

Moving FORWARD along the X-axis...


In [22]:
# Move BACKWARD (X-axis)
for _ in range(10):
    # A negative number moves it backward.
    action = np.array([-1.0, 0.0, 0.0])
    env.step(action)
    time.sleep(0.1)

Moving BACKWARD along the X-axis...


### 3. Y-Axis Movement - Left and Right

In [ ]:
# Move RIGHT (Y-axis)
raise NotImplementedError("Try implementing the Y-axis movement yourself! Remember, action[1] controls the Y-axis.")


In [ ]:
# Move LEFT (Y-axis)
raise NotImplementedError("Try implementing the Y-axis movement yourself! Remember, action[1] controls the Y-axis.")

### 4. Z-Axis Movement - Up and Down

In [31]:
# Move UP (Z-axis)
raise NotImplementedError("Try implementing the Z-axis movement yourself! Remember, action[2] controls the Z-axis.")

In [ ]:
raise NotImplementedError("Try implementing the Z-axis movement yourself! Remember, action[2] controls the Z-axis.")

In [ ]:
env.close()

## Affectors - Input systems/sensors
Where am I? Where is the target?

In [33]:
env = gym.make("PandaReach-v3") # No render mode needed to just read sensors values
obs, info = env.reset()

# The gripper's current [X, Y, Z] coordinate
current_pos = obs["achieved_goal"]

# The target dot's [X, Y, Z] coordinate
target_pos = obs["desired_goal"]

print("\n===========================================================")
print(f"My Gripper is at: X={current_pos[0]:.2f}, Y={current_pos[1]:.2f}, Z={current_pos[2]:.2f}")
print(f"The Target is at: X={target_pos[0]:.2f}, Y={target_pos[1]:.2f}, Z={target_pos[2]:.2f}")

env.close()

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886

My Gripper is at: X=0.04, Y=-0.00, Z=0.20
The Target is at: X=0.11, Y=-0.05, Z=0.19


## Building a simple controller to reach the target
Now that we know how to read the current position of the gripper and the target position, we can build a simple controller that calculates the direction from the current position to the target and generates actions that move the arm in that direction. We need to check the differences between each of the X, Y, and Z coordinates and create an action that moves the arm towards the target by a fix step size.

In [ ]:
env = gym.make("PandaReach-v3", render_mode="human")

In [36]:
obs, info = env.reset()
step_size = 0.05  # How much to move in each step (tune this for faster/slower movement)
for _ in range(100):
    current_x = obs["achieved_goal"][0]
    target_x  = obs["desired_goal"][0]

    # Move forward or backward based on the X coordinate difference
    if round(abs(target_x - current_x), 2) < 0.01:
        move_x = 0.0  # Already close enough, don't move
    elif target_x > current_x:
        move_x = step_size  # Go forward
    else:
        move_x = -step_size # Go backward

    # Apply our calculated move to the X slot, keep others 0
    action = np.array([move_x, 0.0, 0.0])

    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.01)

    # Move Y, Z too
    current_y = obs["achieved_goal"][1]
    target_y  = obs["desired_goal"][1]
    if round(abs(target_y - current_y), 2) < 0.01:
        move_y = 0.0
    elif target_y > current_y:
        move_y = step_size
    else:
        move_y = -step_size

    current_z = obs["achieved_goal"][2]
    target_z  = obs["desired_goal"][2]
    if round(abs(target_z - current_z), 2) < 0.01:
        move_z = 0.0
    elif target_z > current_z:
        move_z = step_size
    else:
        move_z = -step_size

    action = np.array([move_x, move_y, move_z])
    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.05)

    print(f"Current Pos: X={current_x:.2f}, Y={current_y:.2f}, Z={current_z:.2f} \n"
          f"Target Pos: X={target_x:.2f}, Y={target_y:.2f}, Z={target_z:.2f} \n"
          f"Action: X={move_x:.3f}, Y={move_y:.3f}, Z={move_z:.3f} \n"
          f"Reward: {reward:.3f} \n"
          f"Terminated: {terminated} | Truncated: {truncated} \n")

    if terminated:
        print("Reached the target")
        break

    if truncated:
        print("Episode truncated: took more than 50 steps")
        break


argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=5
argv[0] = --unused
argv[1] = --background_color_red=0.8745098114013672
argv[2] = --background_color_green=0.21176470816135406
argv[3] = --background_color_blue=0.1764705926179886
argv[4] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA GeForce RTX 2050/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 610.43.02
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
pthread_getconcurrency()=0
Version = 3.3.0 NVIDIA 610.43.02
Vendor = NVIDIA Corporation
Renderer = NVIDIA GeForce RTX 2050/PCIe/SSE2
b3Pr

In [35]:
env.close()

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed


## Proportional Controller
While the previous controller worked, it was a bit clunky and slow. We can make it smoother and faster by using a proportional controller, which calculates the direction from the current position to the target and generates actions that are proportional to that direction. This way, the arm will move faster when it's far from the target and slow down as it gets closer, allowing for a more efficient and natural movement towards the target position. The proportional controller is a simple yet effective way to control the arm without needing to implement complex algorithms, and it can serve as a good baseline for understanding how to design controllers for robotic arms in reinforcement learning environments.

<img src="https://multispanindia.com/1_1304x542.png" alt="Proportional Controller" width="400" height="200"/>

In [39]:
env = gym.make("PandaReach-v3", render_mode="human")
obs, info = env.reset()

for _ in range(200):
    current_pos = obs["achieved_goal"]
    target_pos  = obs["desired_goal"]

    # Calculate the error (distance to target) for X, Y, and Z
    error_x = target_pos[0] - current_pos[0]
    error_y = target_pos[1] - current_pos[1]
    error_z = target_pos[2] - current_pos[2]

    # Multiply by a speed factor (K)
    K = 10.0
    action = np.array([error_x * K, error_y * K, error_z * K])

    # Safety limit: don't let the motors exceed their -1.0 to 1.0 limit
    action = np.clip(action, -1.0, 1.0)

    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.01)

    if terminated or truncated:
        print("Target Reached!")
        # obs, info = env.reset()
        # time.sleep(2.0)
        break

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=5
argv[0] = --unused
argv[1] = --background_color_red=0.8745098114013672
argv[2] = --background_color_green=0.21176470816135406
argv[3] = --background_color_blue=0.1764705926179886
argv[4] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA GeForce RTX 2050/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 610.43.02
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
pthread_getconcurrency()=0
Version = 3.3.0 NVIDIA 610.43.02
Vendor = NVIDIA Corporation
Renderer = NVIDIA GeForce RTX 2050/PCIe/SSE2
b3Pr

In [40]:
env.close()

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed


At this point you are wondering, "This is great, but how do we get the robot to learn to do this on its own? I don't want to have to write a controller for every single task!"
<br>
Here comes the magic of reinforcement learning.

## What is Reinforcement Learning?
Reinforcement learning (RL) is a field of machine learning that uses **"rewards"** to guide the AI's behavior.
<br>
**The Concept:** Learning through Trial and Error.
**The Explanation:**
Imagine training a puppy. You don’t give the puppy a math equation on how to sit; you wait for it to sit, and then you give it a treat. Reinforcement Learning works exactly the same way. There is no hard-coded script. The AI learns by interacting with the world and trying to get the highest score possible.

It relies on a continuous 4-step cycle:
1. **Agent:** The "Brain" (our AI).
2. **Action:** What the brain decides to do (e.g., move the robot arm left).
3. **Environment:** The physics world (MuJoCo/PyBullet) that reacts to the action.
4. **State & Reward:** The environment tells the brain what the new world looks like (State) and gives it a score (Reward: +1 for good, -1 for bad).

![RL Loop](https://upload.wikimedia.org/wikipedia/commons/1/1b/Reinforcement_learning_diagram.svg)

There are multiple algorithms that can be used to train RL agents, but the most common are:
* **Policy Gradient:** The AI learns to maximize the reward by adjusting its policy (e.g., the way it decides which action to take in a given state).
* **Value Function:** The AI learns to maximize the reward by adjusting its value function (e.g., the way it estimates how good a state or action is).
* **Q-Learning:** The AI learns to maximize the reward by adjusting both the policy and the value function.
* **Deep Q-Learning:** The AI learns to maximize the reward by adjusting both the policy and the value function using a deep neural network.

While there are numerous RL algorithms, we will focus on two of the most popular and effective ones for robotics tasks: **PPO (Proximal Policy Optimization)** and **SAC (Soft Actor-Critic)**.

### 2. PPO (Proximal Policy Optimization): The "Safe" Learner
**The Concept:** Mountain climbing with a safety rope.
**The Explanation:**
Before PPO, RL algorithms had a massive problem: "Catastrophic Forgetting." An AI would spend 5 hours learning how to walk, then suddenly take one bad mathematical step, fall off a cliff, and completely forget how to walk.

OpenAI invented **PPO** to fix this (and used it to train ChatGPT!). The "Proximal" in PPO means "Close." PPO puts mathematical guardrails (called *clipping*) around the AI's brain. It says: *"You can update your brain based on what you just learned, but you can only change your brain a tiny bit at a time."* Because it refuses to take massive, reckless leaps, PPO is incredibly stable and almost always succeeds on straightforward tasks.

 ![PPO Architecture](assets/ppo_actor_critic_architecture.svg)

Learn more about PPO [here](https://spinningup.openai.com/en/latest/algorithms/ppo.html).

### 3. SAC (Soft Actor-Critic): The "Creative Explorer"
**The Concept:** The two-brained explorer.
**The Explanation:**
If PPO is the safe, steady worker, SAC is the mad scientist. SAC is considered the absolute gold standard for complex continuous robotics (like grasping objects). Let's break down the name:
*   **Actor-Critic:** The AI literally has two brains. The **Actor** controls the robot arm. The **Critic** sits back, watches, and gives a harsh rating on how good the Actor's moves are.
*   **Soft (Maximum Entropy):** Standard RL says: *"Maximize your reward."* SAC changes the rules and says: *"Maximize your reward **PLUS** your randomness (entropy)."*

Because SAC is actively rewarded for trying new, crazy, random things, it is much better at solving puzzles. If a robot needs to figure out how to slide its metal fingers under a plastic block to lift it, PPO might get stuck doing the same safe movement forever. SAC will try wildly different angles until it successfully grabs it.

![Actor Critic](assets/sac_twin_critic_architecture.svg)
<br>
Learn more about SAC [here](https://spinningup.openai.com/en/latest/algorithms/sac.html).
For Entropy, check out this great video by 3Blue1Brown:  [https://www.youtube.com/watch?v=l6DKRf-fAAM](https://www.youtube.com/watch?v=l6DKRf-fAAM)

##### Let's train a PPO agent to solve the PandaReachDense-v3 environment!

In [ ]:
env_id = "PandaReachDense-v3"
# We use the Dense version so the AI gets a score based on distance
train_env = gym.make(env_id)

#### Creating a model with Stable Baselines3
We will use the PPO algorithm from Stable Baselines3. SB3 provides a simple interface to create and train RL agents. We will specify the policy type (MultiInputPolicy) and the environment we want to train on. The MultiInputPolicy allows the agent to process both the achieved goal and the desired goal, which is essential for tasks like reaching where the agent needs to understand its current position relative to the target.

Read more SB3 [here](https://stable-baselines3.readthedocs.io/en/master/).

In [4]:
# MultiInputPolicy allows the AI to read our dictionary (achieved/desired goal)
model = PPO("MultiInputPolicy", train_env, verbose=1)

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


#### Training the model
Now that we have a model, we can start training it on the environment. The `learn` method will run the training loop for a specified number of timesteps. During training, the agent will interact with the environment, receive rewards, and update its policy to improve its performance over time. The progress bar will show us how far along the training process is, and we can expect to see the rewards increase as the agent learns to reach the target position more effectively. After training is complete, we will save the model so that we can use it later for testing and evaluation.

In [3]:
print("Starting training... Please wait ~15-60 seconds.") # Depending on your GPU/CPU it may take a while to train
# PPO will easily solve the Dense environment in 25k-30k steps
model.learn(total_timesteps=25_000, progress_bar=True)

# Save the trained model
model.save("models/ppo_panda_reach_dense")
train_env.close()
print("Training complete! Model saved as 'ppo_panda_reach_dense'.")

Starting training... Please wait ~15-30 seconds.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 40       |
|    ep_rew_mean     | -8.98    |
|    success_rate    | 0.235    |
| time/              |          |
|    fps             | 340      |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 41.4        |
|    ep_rew_mean          | -9.44       |
|    success_rate         | 0.224       |
| time/                   |             |
|    fps                  | 266         |
|    iterations           | 2           |
|    time_elapsed         | 15          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.005636972 |
|    clip_fraction        | 0.0426      |
|    clip_range           | 0.2

#### Model Evaluation
Now that we have trained the model, we want to see how well it performs in the environment. We will create a new environment with graphics turned on so we can visually observe the agent's behavior. We will load the trained model and run a loop where the agent takes actions based on its learned policy. This evaluation will allow us to see the results of our training and how effectively the agent has learned to control the robotic arm to achieve its goal.

In [ ]:
# Create a new environment for testing, with graphics turned on to see the agent in action
test_env = gym.make("PandaReachDense-v3", render_mode="human")

In [9]:
# Load the trained model
model = PPO.load("models/ppo_panda_reach_dense")

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=5
argv[0] = --unused
argv[1] = --background_color_red=0.8745098114013672
argv[2] = --background_color_green=0.21176470816135406
argv[3] = --background_color_blue=0.1764705926179886
argv[4] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA GeForce RTX 2050/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 610.43.02
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
pthread_getconcurrency()=0
Version = 3.3.0 NVIDIA 610.43.02
Vendor = NVIDIA Corporation
Renderer = NVIDIA GeForce RTX 2050/PCIe/SSE2
b3Pr

In [32]:
# Test the trained model
obs, info = test_env.reset()
for _ in range(500):
    # Predict the next action based on the current observation, deterministic=True means we want the best action, not a random one
    action, _states = model.predict(obs, deterministic=True)

    obs, reward, terminated, truncated, info = test_env.step(action)

    print(f"Reward: {reward:.2f} | Terminated: {terminated} | Truncated: {truncated}")
    if terminated:
        print("RL Reached the Target!")
        break

    elif truncated:
        print("Episode truncated: took more than 50 steps")
        break

    time.sleep(0.01)
# Rerun this cell to see the trained agent in action again!

Reward: -0.13 | Terminated: False | Truncated: False
Reward: -0.07 | Terminated: False | Truncated: False
Reward: -0.02 | Terminated: True | Truncated: False
AI Reached the Target!


In [8]:
# Close the test environment when done
test_env.close()

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed


#### Let's Load some pretrained models and see how they perform!
We have pre-trained PPO Model on the PandaReachDense-v3 environment

In [ ]:
# Load the backup
backup_model = PPO.load("models/ppo_reach_backup.zip")

test_env = gym.make("PandaReachDense-v3", render_mode="human")

obs, info = test_env.reset()

for _ in range(500):
    action, _states = backup_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = test_env.step(action)

    if terminated or truncated:
        obs, info = test_env.reset()
        time.sleep(1.0)

    time.sleep(0.01)

test_env.close()

#### Now your turn!
Try to train a PPO agent on the `PandaPickAndPlace-v3` environment. This environment is more complex than `PandaReach` because the agent needs to learn not only how to reach a target position but also how to pick up an object and place it at a target location. The state space and action space are more complex, and the reward structure is different, which will require the agent to learn more sophisticated behaviors. You can use the same approach as we did for `PandaReach`, but you may need to adjust the training parameters (like the number of timesteps) to allow the agent enough time to learn the task effectively. Good luck, and have fun experimenting with this new environment!

Try and understand the environment by inspecting the observation space and action space.

In [ ]:
env_id = "PandaPickAndPlace-v3"
env = gym.make(...)
s_size = ...

print("\n===========================================================")
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", ...)

a_size = ...
action = ...
print("===========================================================")
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Random Action Sampled:", action)

Now that you have a better understanding of the environment, try to train a PPO agent on the `PandaPickAndPlace-v3` environment. Remember to adjust the training parameters as needed, and don't forget to evaluate your trained model to see how well it performs in the environment.

In [ ]:
# Create the environment
train_env = gym.make(...)

# Create the PPO model
model = PPO(...)

In [ ]:
# Train the model
model.learn(total_timesteps=..., progress_bar=True)

# Save the trained model
model.save("models/ppo_panda_pick_and_place")
train_env.close()
print("Training complete! Model saved as 'ppo_panda_pick_and_place'.")

In [ ]:
# Create a new environment for testing
test_env = gym.make(...)

In [ ]:
# Load the trained model
model.load(...)

In [ ]:
# Test the trained model
obs, info = test_env.reset()
for _ in range(500):
    raise NotImplementedError("Try implementing the testing loop yourself! Use the same structure as we did for PandaReach, but make sure to use the correct environment and model for PandaPickAndPlace.")
    # 1. Predict the next action based on the current observation

    # 2. Take the action in the environment and observe the new state and reward

    # 3. Print the reward

    # 4. Check if the episode has terminated or truncated, and reset if necessary

    # 5. Sleep for a short time to simulate real-time behavior

# Close the test environment when done
test_env.close()

#### What did you observe during training and testing? Did the agent learn to pick and place the object successfully?

Probably didn't work well, don't worry! Let's try with SAC. We won't go through the training process again, but you can try it on your own by following the same steps as we did for PPO, but using the SAC algorithm instead.
<br>
__Let's load a pre-trained SAC model and see how it performs on the PandaPickAndPlace-v3 environment!__

In [1]:
env = gym.make("PandaPickAndPlace-v3", render_mode="human")
model = SAC.load("sac_pick_and_place.zip")

obs, info = env.reset()

for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        obs, info = env.reset()
        time.sleep(0.5)

    time.sleep(0.01)

env.close()